# Pandas Fundamentals
**Data Analysis**

---

## What is Pandas?

Pandas is a Python library for working with **tabular data** — rows and columns, just like a spreadsheet or a database table.

Two core structures:
- **Series** — one column with labels
- **DataFrame** — a full table (multiple Series sharing the same row index)

Every ML project starts here: load raw data → clean → transform → pass to a model.

In [1]:
import pandas as pd
import numpy as np

---
## 1. Series — One Labeled Column

A Series is like a Python list, but every value has a **label** (the index).
That label lets you access values by name instead of by position.

The labels can be integer, float, strings, date or anything.

In [2]:
# by default the index are 0, 1, 2... integers
s = pd.Series([20, 30, 35, 36, 98])
print(s)

# From a dict — keys become labels (index)
scores = pd.Series({"Alice": 88, "Bob": 92, "Charlie": 75, "Diana": 95, "Eve": 60})

print(scores)
print(scores["Alice"])         # access by label
print(scores[scores > 80])     # filter — returns rows where condition is True

0    20
1    30
2    35
3    36
4    98
dtype: int64
Alice      88
Bob        92
Charlie    75
Diana      95
Eve        60
dtype: int64
88
Alice    88
Bob      92
Diana    95
dtype: int64


---
## 2. DataFrame — The Table

In [3]:
data = {
    "name":       ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "age":        [25, 30, 35, 28, 22],
    "department": ["Eng", "Marketing", "Eng", "HR", "Eng"],
    "salary":     [90000, 72000, 105000, 68000, 88000],
    "years_exp":  [2, 5, 10, 3, 1],
}
df = pd.DataFrame(data)
df

,name,age,department,salary,years_exp
0,Alice,25,Eng,90000,2
1,Bob,30,Marketing,72000,5
2,Charlie,35,Eng,105000,10
3,Diana,28,HR,68000,3
4,Eve,22,Eng,88000,1


In [4]:
# Always run these first on any new dataset
print(df.shape)      # (rows, columns)
df.info()            # column names, data types, non-null counts
df.describe()      # stats summary for numeric columns (mean, std, min, max)
df.head()          # first 5 rows

(5, 5)
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name        5 non-null      str  
 1   age         5 non-null      int64
 2   department  5 non-null      str  
 3   salary      5 non-null      int64
 4   years_exp   5 non-null      int64
dtypes: int64(3), str(2)
memory usage: 332.0 bytes


,name,age,department,salary,years_exp
0,Alice,25,Eng,90000,2
1,Bob,30,Marketing,72000,5
2,Charlie,35,Eng,105000,10
3,Diana,28,HR,68000,3
4,Eve,22,Eng,88000,1


---
## 3. ⚠️ loc vs iloc — The #1 Confusion in Pandas

Pandas has **two indexing systems** and mixing them up is the most common mistake.

### `loc` — label-based
You refer to rows and columns by their **names/labels**.
```python
df.loc[2, "salary"]         # row with label 2, column named 'salary'
df.loc[0:2, "salary"]       # rows with labels 0, 1, 2 — INCLUSIVE on both ends
```

### `iloc` — position-based
You refer to rows and columns by their **integer position** (like a Python list).
```python
df.iloc[2, 3]               # 3rd row, 4th column — by position
df.iloc[0:2, 0:3]           # rows at position 0,1 — EXCLUSIVE end (standard Python)
```

### ⚠️ The critical difference — slicing end behavior

| | `loc[0:2]` | `iloc[0:2]` |
|---|---|---|
| Returns | rows 0, 1, **2** (inclusive) | rows 0, **1** only (exclusive) |

This trips everyone up because `loc` breaks the normal Python slicing rule.

### ⚠️ The deeper problem — labels vs positions diverge after filtering
```python
label   name    dept
  0     Alice   Eng
  1     Bob     Mkt      ← Bob gets removed by filter
  2     Charlie Eng
  3     Diana   HR       ← Diana gets removed by filter
  4     Eve     Eng


After filtering df[df["dept"] == "Eng"]:

label   name             position
  0     Alice    ← still label 0, position 0
  2     Charlie  ← still label 2, position 1   ← GAP HERE
  4     Eve      ← still label 4, position 2   ← GAP HERE
Now:

iloc[1] → position 1 → Charlie ✅
loc[1] → label 1 → KeyError — Bob had label 1, but Bob is gone ❌


Simple rule to remember:
After filtering, labels have gaps. Positions never do.

Use iloc when you mean "give me the Nth row."
Use loc when you mean "give me the row with this specific label."
```

In [5]:
# loc — label-based (inclusive end)
print(df.loc[0:2, ["name", "salary"]])   # rows 0, 1, 2 included

      name  salary
0    Alice   90000
1      Bob   72000
2  Charlie  105000


In [6]:
# iloc — position-based (exclusive end, like Python lists)
print(df.iloc[0:2, 0:2])                 # rows at position 0, 1 only

    name  age
0  Alice   25
1    Bob   30


In [7]:
# The label vs position gap — see it happen
engineers = df[df["department"] == "Eng"]
print("Index labels:", engineers.index.tolist())   # [0, 2, 4] — not 0, 1, 2!

print(engineers.iloc[1])            # position 1 → Charlie ✅
# print(engineers.loc[1])           # label 1 → KeyError ❌ (Bob was filtered out)

Index labels: [0, 2, 4]
name          Charlie
age                35
department        Eng
salary         105000
years_exp          10
Name: 2, dtype: object


---
## 4. ⚠️ Boolean Indexing — Filtering Rows

The main way to filter rows. Works in two steps:
1. A condition on a column produces a **boolean Series** (True/False for each row)
2. Pass that into `df[...]` — only `True` rows are returned

### ⚠️ Never use `and` / `or` — use `&` / `|`

When combining two conditions, Python's `and`/`or` doesn't work:
```python
# WRONG — raises ValueError: The truth value of a Series is ambiguous
df[df["dept"] == "Eng" and df["salary"] > 80000]

# RIGHT — & does element-wise comparison across the whole column
df[(df["dept"] == "Eng") & (df["salary"] > 80000)]
```

Why? `and` tries to evaluate the entire Series as one True/False — which is ambiguous (is a Series with [True, False, True] "true" overall?). `&` compares row-by-row.

### ⚠️ Always wrap each condition in parentheses

```python
# WRONG — operator precedence makes & bind before ==, gives wrong result
# What you wrote
df["dept"] == "Eng" & df["salary"] > 80000

# What Python actually sees (& runs first)
df["dept"] == ("Eng" & df["salary"]) > 80000
#                ↑
#         tries to do & between a string and a column → error


# RIGHT
# What you want Python to see
(df["dept"] == "Eng") & (df["salary"] > 80000)
#       ↑                      ↑
#  evaluated first         evaluated first
#              then & combines the two results

Simple rule: whenever you combine conditions with & or | , always wrap each condition in its own parentheses.
```

In [8]:
# Step 1: condition → boolean Series
mask = df["department"] == "Eng"
print(mask.tolist())

[True, False, True, False, True]


In [9]:
# Step 2: filter using the mask
print(df[mask])

      name  age department  salary  years_exp
0    Alice   25        Eng   90000          2
2  Charlie   35        Eng  105000         10
4      Eve   22        Eng   88000          1


In [10]:
# Combining conditions — & (AND), | (OR), ~ (NOT)
senior_eng = df[(df["department"] == "Eng") & (df["years_exp"] >= 5)]
print(senior_eng)

print("*" * 20)
# NOT — ~ flips the mask
not_eng = df[~(df["department"] == "Eng")]
print(not_eng)

      name  age department  salary  years_exp
2  Charlie   35        Eng  105000         10
********************
    name  age department  salary  years_exp
1    Bob   30  Marketing   72000          5
3  Diana   28         HR   68000          3


In [11]:
# .query() is a cleaner way to write the same filter
df.query("department == 'Eng' and years_exp >= 5")

,name,age,department,salary,years_exp
2,Charlie,35,Eng,105000,10


---
## 5. ⚠️ Vectorization — No Loops in Pandas

In regular Python, you'd loop over rows to compute something for each one.
In Pandas, **you never loop over rows** — you apply operations to the whole column at once.

```python
# Python loop way — works but very slow (pure Python, row by row)
for i in range(len(df)):
    df.loc[i, "revenue"] = df.loc[i, "units"] * df.loc[i, "price"]

# Pandas way — vectorized (runs in optimized C, 100x+ faster)
df["revenue"] = df["units"] * df["price"]
```

Pandas passes column operations down to NumPy, which runs them in compiled C code.
On a million rows: loop ≈ seconds, vectorized ≈ milliseconds.

**Rule of thumb:** if you're writing a `for` loop over DataFrame rows, you're doing it wrong — stop and find the vectorized equivalent.

In [12]:
# All of these are vectorized — applied to every row with no loop
df["salary_per_exp"] = df["salary"] / df["years_exp"]   # math on two columns
df["is_senior"]      = df["years_exp"] >= 5              # boolean column
df["name_lower"]     = df["name"].str.lower()            # string operation

df[["name", "salary", "salary_per_exp", "is_senior", "name_lower"]]

,name,salary,salary_per_exp,is_senior,name_lower
0,Alice,90000,45000.000000,False,alice
1,Bob,72000,14400.000000,True,bob
2,Charlie,105000,10500.000000,True,charlie
3,Diana,68000,22666.666667,False,diana
4,Eve,88000,88000.000000,False,eve


---
## 6. ⚠️ Missing Values (NaN) — Handle Before Everything Else

Real data always has gaps. Pandas represents them as `NaN` (Not a Number).

### ⚠️ NaN is silent and contagious

```python
np.nan + 5          # → nan   (no error, just wrong result)
np.nan > 10         # → False (no error, just wrong result)
series.mean()            # → int  (.mean() skips NaN by default — silently ignores it)
series * 2               # → NaN spreads into the result for that row
```

This is dangerous because you get no error — just silently wrong results. An ML model trained on a column with NaN values will either crash or produce garbage predictions.

### Strategy: check → decide → fill or drop

1. **Check first:** `df.isnull().sum()` — see which columns have how many missing values
2. **Fill** (`fillna`) — replace NaN with something sensible
   - Numbers → use **median** (not mean — median isn't skewed by outliers)
   - Categories → use `"Unknown"` or the most frequent value
3. **Drop** (`dropna`) — remove rows with NaN — only if missingness is rare and you have plenty of data

Always `.copy()` the DataFrame before modifying — otherwise you risk accidentally changing the original.

In [13]:
# Introduce NaN and see the contagion
df_missing = df[["name", "salary", "department"]].copy()   # .copy() — don't modify original
df_missing.loc[1, "salary"] = None
df_missing.loc[3, "department"] = None

print(df_missing.isnull().sum())                             # check: salary=1, department=1

# .mean() skips NaN by default — still gives a number (averages the other 4 rows)
print("Mean salary:", df_missing["salary"].mean())           # → 87750.0, NOT nan

# Contagion shows up in arithmetic — NaN in one column spreads to the result
df_missing["salary_x2"] = df_missing["salary"] * 2
print(df_missing[["name", "salary", "salary_x2"]])           # Bob's row → NaN

df_missing.drop(columns=["salary_x2"], inplace = True)
# Also: scikit-learn will crash entirely if NaN exists in input — ValueError: Input contains NaN

name          0
salary        1
department    1
dtype: int64
Mean salary: 87750.0
      name    salary  salary_x2
0    Alice   90000.0   180000.0
1      Bob       NaN        NaN
2  Charlie  105000.0   210000.0
3    Diana   68000.0   136000.0
4      Eve   88000.0   176000.0


```python
How .copy() works
When you do df2 = df or df2 = df[["col1", "col2"]], pandas doesn't create new data — it creates a reference pointing to the same memory as the original. Modifying df2 can modify df as well.

.copy() forces pandas to create a completely independent copy in memory. Changes to the copy never touch the original.

Without .copy() — dangerous:


df2 = df[["name", "salary"]]   # might be a view — shares memory with df
df2.loc[1, "salary"] = None    # could silently modify df too!
                                # also raises SettingWithCopyWarning
With .copy() — safe:


df2 = df[["name", "salary"]].copy()   # independent copy — own memory
df2.loc[1, "salary"] = None           # only df2 changes, df is untouched
Why pandas has this behavior
Pandas tries to avoid duplicating data unnecessarily (data can be huge — millions of rows). So slices and filters often return a view into the original rather than new data. It's a memory optimization.

The problem is pandas doesn't always guarantee whether you got a view or a copy — it depends on the operation. So instead of guessing, just always .copy() when you intend to modify a subset. It's a safety habit.

Simple rule: if you're going to modify a DataFrame that came from slicing or filtering another one, always .copy() it first.
```

In [14]:
df_fixed = df_missing.copy()

# Fill numbers with median, categories with a placeholder
df_fixed["salary"] = df_fixed["salary"].fillna(df_fixed["salary"].median())

df_fixed["department"] = df_fixed["department"].fillna("Unknown")

print(df_fixed.isnull().sum())                    # all zeros — clean
print("Mean salary:", df_fixed["salary"].mean())  # works now

name          0
salary        0
department    0
dtype: int64
Mean salary: 88000.0


---
## 7. Data Loading — Brief

CSV is the most common format. A few key options to know:

In [15]:
df.to_csv("employees.csv", index=False)   # index=False: don't write row numbers to file

pd.read_csv("employees.csv")              # basic load

pd.read_csv(
    "employees.csv",
    usecols=["name", "salary"],    # only load columns you need — saves memory on large files
    dtype={"salary": float},       # tell pandas the type — avoids wrong type guesses
)

,name,salary
0,Alice,90000.0
1,Bob,72000.0
2,Charlie,105000.0
3,Diana,68000.0
4,Eve,88000.0


---
## 8. Basic Operations — Brief

In [16]:
df[["name", "salary"]].sort_values("salary", ascending=False)  # sort — returns new df

,name,salary
2,Charlie,105000
0,Alice,90000
4,Eve,88000
1,Bob,72000
3,Diana,68000


In [17]:
df.rename(columns={"years_exp": "experience"})   # rename columns

,name,age,department,salary,experience,salary_per_exp,is_senior,name_lower
0,Alice,25,Eng,90000,2,45000.000000,False,alice
1,Bob,30,Marketing,72000,5,14400.000000,True,bob
2,Charlie,35,Eng,105000,10,10500.000000,True,charlie
3,Diana,28,HR,68000,3,22666.666667,False,diana
4,Eve,22,Eng,88000,1,88000.000000,False,eve


In [18]:
df.drop(columns=["salary_per_exp", "is_senior", "name_lower"])  # remove columns

,name,age,department,salary,years_exp
0,Alice,25,Eng,90000,2
1,Bob,30,Marketing,72000,5
2,Charlie,35,Eng,105000,10
3,Diana,28,HR,68000,3
4,Eve,22,Eng,88000,1


In [19]:
df["department"].value_counts()   # frequency count per unique value — useful for spotting imbalance

department
Eng          3
Marketing    1
HR           1
Name: count, dtype: int64

## .str Accessor — String Operations on a Column

`.str` lets you apply string methods to an entire column without a loop.

### Common ones:
```python
df["col"].str.lower()                      # lowercase all values
df["col"].str.upper()                      # uppercase all values
df["col"].str.strip()                      # remove leading/trailing whitespace
df["col"].str.contains("x", case=False)   # check if value contains a pattern
df["col"].str.startswith("A")             # check if value starts with something
df["col"].str.replace("old", "new")       # replace text
df["col"].str.len()                        # length of each string
⚠️ .str.contains() returns a boolean mask — not the filtered rows

# Wrong — this just prints True/False for each row
sales["product"].str.contains("e", case=False)

# Right — pass the mask into df[...] to get the actual rows
sales[sales["product"].str.contains("e", case=False)]

---
## 9. Practice

In [20]:
sales = pd.DataFrame({
    "product": ["Widget", "Gadget", "Widget", "Doohickey", "Gadget", "Widget", "Doohickey"],
    "region":  ["North", "South", "South", "North", "North", "East", "East"],
    "units":   [150, 200, 175, 90, 210, 130, 80],
    "price":   [29.99, 49.99, 29.99, 19.99, 49.99, 29.99, 19.99],
})
sales

,product,region,units,price
0,Widget,North,150,29.99
1,Gadget,South,200,49.99
2,Widget,South,175,29.99
3,Doohickey,North,90,19.99
4,Gadget,North,210,49.99
5,Widget,East,130,29.99
6,Doohickey,East,80,19.99


In [22]:
# 1. Add a 'revenue' column (units * price) — do it vectorized, no loop
sales["revenue"] = sales["units"] * sales["price"]
print(sales["revenue"])

0     4498.50
1     9998.00
2     5248.25
3     1799.10
4    10497.90
5     3898.70
6     1599.20
Name: revenue, dtype: float64


In [25]:
# 2. Filter rows where product is 'Gadget' AND region is 'North'
#    Think: which operator do you use to combine conditions?

sales_filtered = sales[(sales["product"] == "Gadget") & (sales["region"] == "North")]

print(sales_filtered)


  product region  units  price  revenue
4  Gadget  North    210  49.99  10497.9


In [27]:
# 3. sales.loc[1:3] vs sales.iloc[1:3] — how many rows does each return?
#  loc is a label based and slicing is inclusive on both ends: so it returns row with label 1, 2, 3
# iloc is position based and slicing is exclusing on last end: so it returns row with position 1, 2

print(sales.loc[1:3])

print("*" * 20)

print(sales.iloc[1:3])

     product region  units  price  revenue
1     Gadget  South    200  49.99  9998.00
2     Widget  South    175  29.99  5248.25
3  Doohickey  North     90  19.99  1799.10
********************
  product region  units  price  revenue
1  Gadget  South    200  49.99  9998.00
2  Widget  South    175  29.99  5248.25


In [30]:
# 4. Set units for rows 2 and 4 to NaN. Check missing count. Fill with median.
sales.loc[2, "units"] = np.nan
sales.loc[4, "units"] = np.nan

#check missing
print(sales.isnull().sum())

# fill with median
sales["units"] = sales["units"].fillna(sales["units"].median())
print(sales["units"])

product    0
region     0
units      2
price      0
revenue    0
dtype: int64
0    150.0
1    200.0
2    130.0
3     90.0
4    130.0
5    130.0
6     80.0
Name: units, dtype: float64


In [32]:
# 5. Filter rows where product name contains the letter 'e' (case-insensitive)
sales_filtered2 = sales[sales["product"].str.contains("e", case=False)]
print(sales_filtered2)


     product region  units  price   revenue
0     Widget  North  150.0  29.99   4498.50
1     Gadget  South  200.0  49.99   9998.00
2     Widget  South  130.0  29.99   5248.25
3  Doohickey  North   90.0  19.99   1799.10
4     Gadget  North  130.0  49.99  10497.90
5     Widget   East  130.0  29.99   3898.70
6  Doohickey   East   80.0  19.99   1599.20


---
## Quick Reference

```python
# ── INSPECT ────────────────────────────────────────────────────
df.shape                      # (rows, cols)
df.info()                     # types + null counts
df.describe()                 # stats for numeric cols
df.isnull().sum()             # missing per column

# ── SELECT ─────────────────────────────────────────────────────
df["col"]                     # one column → Series
df[["col1", "col2"]]          # multiple columns → DataFrame
df.loc[label, "col"]          # by label — slice is INCLUSIVE
df.iloc[pos, 0]               # by position — slice is EXCLUSIVE

# ── FILTER ─────────────────────────────────────────────────────
df[df["col"] > val]           # single condition
(df["a"] > 1) & (df["b"] == "x")   # AND — use &, not 'and'
(df["a"] > 1) | (df["b"] == "x")   # OR  — use |, not 'or'
~(df["col"] == "x")           # NOT — use ~

# ── MODIFY (vectorized) ────────────────────────────────────────
df["new"] = df["a"] * df["b"] # add column
df["col"].fillna(df["col"].median())  # fill NaN
df.rename(columns={"old": "new"})
df.drop(columns=["col"])
df.sort_values("col", ascending=False)

# ── STRING OPS ─────────────────────────────────────────────────
df["col"].str.lower()
df["col"].str.contains("x", case=False)   # → boolean mask
df["col"].str.strip()         # remove whitespace
```

---
**Tomorrow (Day 30):** GroupBy & Aggregation